# Exp-4b — Rerank with HyDE passages (register-matched query)

**Why this exp.** Exp-4 reranked with the raw German query and lost recall at every K (`@50` 0.161 → 0.114). Diagnosis: the cross-encoder sees `(factual scenario) vs (formal statute)` — the same cross-register gap HyDE+Enum already worked around on the retrieval side. The reranker has no access to those expansions and re-imposes the mismatch.

**Hypothesis.** Feeding HyDE passages (already in formal German statute style) as the reranker's query side will eliminate the register mismatch.

**Design.**
- Same candidate pool as Exp-4: hyde+enum top-1000 per query (reused from `rerank_scores_A4.npz`).
- Reranker: `BAAI/bge-reranker-v2-m3` (fp16).
- For each query, score 3 HyDE passages × 1000 docs = 3000 pairs. Aggregate per (query, doc) with **mean** and **max**.
- Compare against:
  - baseline RRF (no rerank, from Exp-4)
  - rerank with raw DE query (from Exp-4)
  - rerank with HyDE-mean (this exp)
  - rerank with HyDE-max (this exp)

**Gate.** `stat_recall@50 >= 0.20` would be a clear win over baseline (0.161). `>= 0.30` would actually be useful for a sub-25-citation submission. If even register-matched rerank can't beat the baseline RRF, cross-encoder reranking is dead for this task and we move all effort to court-corpus retrieval.

In [1]:
# --- Cell 1. Install deps ---
!pip install -qU numpy==1.26.4
!pip install -qU FlagEmbedding pandas numpy
# Restart runtime after installing numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 162.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0,

In [1]:
# --- Cell 2. Mount Drive & paths ---
from google.colab import drive
drive.mount('/content/drive')

import json, pickle, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/swiss_law/data')
ART  = ROOT / 'artifacts'
for f in ['laws_bgem3.npy', 'exp_A3_expansions.json', 'rerank_scores_A4.npz', 'exp_A4_report.json']:
    assert (ART / f).exists(), f'missing {f} — run Exp-3/4 first'

assert torch.cuda.is_available(), 'need GPU'
print('GPU:', torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
# --- Cell 3. Load data + reuse Exp-4 candidates ---
val  = pd.read_csv(ROOT / 'val.csv')
laws = pd.read_csv(ROOT / 'laws_de.csv')
with open(ART / 'exp_A3_expansions.json', 'r', encoding='utf-8') as f:
    exp = json.load(f)

cits = laws['citation'].tolist()
docs = (laws['citation'].fillna('') + ' | ' +
        laws['title'].fillna('')     + ' | ' +
        laws['text'].fillna('')).tolist()

saved = np.load(ART / 'rerank_scores_A4.npz', allow_pickle=True)
cands = saved['cands']                       # (10, 1000)
scores_de_old = saved['scores_de']           # for comparison
query_ids = [str(x) for x in saved['query_ids']]
assert query_ids == list(val['query_id']), 'order mismatch'

# Sanity-check what fields exp_A3_expansions.json provides
qid0 = query_ids[0]
print('expansion keys for', qid0, ':', list(exp[qid0].keys()))
for k, v in exp[qid0].items():
    if isinstance(v, list):
        print(f'  {k}: list of {len(v)} | first: {str(v[0])[:120]!r}')
    else:
        print(f'  {k}: {type(v).__name__} | {str(v)[:120]!r}')

expansion keys for val_001 : ['query_en', 'query_de', 'hyde', 'enum']
  query_en: str | 'May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of col'
  query_de: str | 'Kann ein Gericht eine dreimonatige Verlängerung der Untersuchungshaft gemäß Art. 221 Abs. 1 lit. b StPO (Kollusionsgefah'
  hyde: str | 'Art. 221 Abs. 1 lit. b StPO ist dahin auszulegen, dass eine Verlängerung der Untersuchungshaft um drei Monate nur dann r'
  enum: str | 'Art. 107 Abs. 1 StPO — Pre-trial detention conditions  \nArt. 108 Abs. 1 StPO — Justification for detention  \nArt. 109 Ab'


In [3]:
# --- Cell 4. Locate HyDE passages, build per-query passage list ---
# Auto-detect the HyDE field. In Exp-2/3 it's typically 'hyde_passages' (list[str], len=3)
# or 'hyde' / 'hyde_text'. Adapt if needed.
candidate_fields = ['hyde_passages', 'hyde', 'hyde_text', 'hypothetical', 'passages']
HYDE_KEY = next((k for k in candidate_fields if k in exp[query_ids[0]]), None)
assert HYDE_KEY is not None, f'no HyDE field found in {list(exp[query_ids[0]].keys())}'
print('HyDE field:', HYDE_KEY)

hyde_per_query = []
for qid in query_ids:
    h = exp[qid][HYDE_KEY]
    if isinstance(h, str):
        # Some pipelines store HyDE as one big string with paragraph breaks
        parts = [p.strip() for p in re.split(r'\n\s*\n', h) if p.strip()]
        hyde_per_query.append(parts[:3] if len(parts) >= 3 else [h])
    else:
        hyde_per_query.append([str(p).strip() for p in h])
    print(f'{qid}: {len(hyde_per_query[-1])} passages, '
          f'first len={len(hyde_per_query[-1][0])}')

HyDE field: hyde
val_001: 3 passages, first len=492
val_002: 3 passages, first len=574
val_003: 3 passages, first len=498
val_004: 3 passages, first len=490
val_005: 3 passages, first len=522
val_006: 3 passages, first len=559
val_007: 3 passages, first len=487
val_008: 3 passages, first len=557
val_009: 3 passages, first len=539
val_010: 3 passages, first len=496


In [4]:
# --- Cell 5. Load reranker ---
!pip install -q "transformers<4.45.0"
from FlagEmbedding import FlagReranker
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

In [5]:
# --- Cell 6. Rerank: each HyDE passage × 1000 candidates per query ---
N_q = len(query_ids)
MAX_PASS = 3   # cap passages per query for predictable cost

# scores_per_passage[i] -> (n_pass_i, 1000)
scores_per_passage = []
t0 = time.time()
for i, qid in enumerate(query_ids):
    passages = hyde_per_query[i][:MAX_PASS]
    sp = np.zeros((len(passages), 1000), dtype=np.float32)
    for p_idx, p in enumerate(passages):
        pairs = [[p, docs[cands[i, j]]] for j in range(1000)]
        s = reranker.compute_score(pairs, batch_size=32, max_length=1024, normalize=True)
        sp[p_idx] = np.asarray(s, dtype=np.float32)
    scores_per_passage.append(sp)
    print(f'  [{i+1}/{N_q}] {qid}: {len(passages)} passages | elapsed {time.time()-t0:.0f}s')
print(f'rerank total: {time.time()-t0:.0f}s')

# Aggregate two ways
scores_hyde_mean = np.stack([sp.mean(axis=0) for sp in scores_per_passage], axis=0)  # (10,1000)
scores_hyde_max  = np.stack([sp.max(axis=0)  for sp in scores_per_passage], axis=0)

np.savez(ART / 'rerank_scores_A4b.npz',
         cands=cands,
         scores_hyde_mean=scores_hyde_mean,
         scores_hyde_max=scores_hyde_max,
         query_ids=np.array(query_ids))

Compute Scores: 100%|██████████| 32/32 [00:01<00:00, 30.39it/s]


  [1/10] val_001: 3 passages | elapsed 5s


Compute Scores: 100%|██████████| 32/32 [00:01<00:00, 21.29it/s]


  [2/10] val_002: 3 passages | elapsed 10s


Compute Scores: 100%|██████████| 32/32 [00:00<00:00, 33.12it/s]


...

  [8/10] val_008: 3 passages | elapsed 33s


Compute Scores: 100%|██████████| 32/32 [00:01<00:00, 22.36it/s]


  [9/10] val_009: 3 passages | elapsed 38s


Compute Scores: 100%|██████████| 32/32 [00:01<00:00, 22.82it/s]

  [10/10] val_010: 3 passages | elapsed 42s
rerank total: 42s


In [6]:
# --- Cell 7. Evaluate vs baselines ---
def parse(s): return [c.strip() for c in str(s).split(';') if c.strip()]
def is_stat(c):
    return not (c.startswith('BGE ') or re.match(r'\d[A-Z]_', c) or re.match(r'[A-Z]\d[A-Z]_', c))

def eval_at_k(order_per_q, label, ks=(10, 20, 30, 50, 100, 200, 500)):
    per_q = []
    for i, row in enumerate(val.itertuples()):
        gold = {c for c in parse(row.gold_citations) if is_stat(c)}
        ranked_docs = cands[i, order_per_q[i]]
        ranked_cits = [cits[j] for j in ranked_docs]
        entry = {'query_id': row.query_id, 'n_gold': len(gold)}
        for k in ks:
            entry[f'hit@{k}'] = len(gold & set(ranked_cits[:k]))
        per_q.append(entry)
    agg = {f'stat_recall@{k}': sum(p[f'hit@{k}'] for p in per_q)
                              / max(1, sum(p['n_gold'] for p in per_q)) for k in ks}
    print(f'=== {label} ===')
    for k, v in agg.items():
        print(f'  {k} = {v:.3f}')
    return {'agg': agg, 'per_query': per_q}

baseline_order = np.tile(np.arange(1000), (N_q, 1))
order_de_old   = np.argsort(-scores_de_old, axis=1)
order_h_mean   = np.argsort(-scores_hyde_mean, axis=1)
order_h_max    = np.argsort(-scores_hyde_max, axis=1)

report = {
    'baseline_rrf':     eval_at_k(baseline_order, 'baseline RRF (no rerank)'),
    'rerank_de_query':  eval_at_k(order_de_old,   'rerank with raw DE query (Exp-4 repro)'),
    'rerank_hyde_mean': eval_at_k(order_h_mean,   'rerank with HyDE passages (mean)'),
    'rerank_hyde_max':  eval_at_k(order_h_max,    'rerank with HyDE passages (max)'),
}

=== baseline RRF (no rerank) ===
  stat_recall@10 = 0.047
  stat_recall@20 = 0.101
  stat_recall@30 = 0.121
  stat_recall@50 = 0.161
  stat_recall@100 = 0.255
  stat_recall@200 = 0.315
  stat_recall@500 = 0.376
=== rerank with raw DE query (Exp-4 repro) ===
  stat_recall@10 = 0.047
  stat_recall@20 = 0.087
  stat_recall@30 = 0.101
  stat_recall@50 = 0.114
  stat_recall@100 = 0.148
  stat_recall@200 = 0.215
  stat_recall@500 = 0.342
=== rerank with HyDE passages (mean) ===
  stat_recall@10 = 0.087
  stat_recall@20 = 0.114
  stat_recall@30 = 0.134
  stat_recall@50 = 0.174
  stat_recall@100 = 0.242
  stat_recall@200 = 0.315
  stat_recall@500 = 0.369
=== rerank with HyDE passages (max) ===
  stat_recall@10 = 0.074
  stat_recall@20 = 0.114
  stat_recall@30 = 0.134
  stat_recall@50 = 0.174
  stat_recall@100 = 0.242
  stat_recall@200 = 0.302
  stat_recall@500 = 0.369


In [7]:
# --- Cell 8. Per-query breakdown @50 ---
print(f'{"qid":<10} {"gold":>4} | {"base":>4} {"deQ":>4} {"hMean":>5} {"hMax":>4} | {"hMean@30":>8} {"hMean@20":>8} {"hMean@10":>8}')
print('-' * 80)
for i, row in enumerate(val.itertuples()):
    qid = row.query_id
    b = report['baseline_rrf']['per_query'][i]
    d = report['rerank_de_query']['per_query'][i]
    m = report['rerank_hyde_mean']['per_query'][i]
    x = report['rerank_hyde_max']['per_query'][i]
    print(f'{qid:<10} {b["n_gold"]:>4} | '
          f'{b["hit@50"]:>4} {d["hit@50"]:>4} {m["hit@50"]:>5} {x["hit@50"]:>4} | '
          f'{m["hit@30"]:>8} {m["hit@20"]:>8} {m["hit@10"]:>8}')

qid        gold | base  deQ hMean hMax | hMean@30 hMean@20 hMean@10
--------------------------------------------------------------------------------
val_001      19 |    6    5     5    5 |        5        5        4
val_002      20 |    3    1     3    2 |        2        1        1
val_003      24 |    3    2     4    4 |        3        3        2
val_004       9 |    2    3     2    2 |        2        2        2
val_005       6 |    2    1     2    2 |        2        1        0
val_006      11 |    4    0     1    1 |        1        1        1
val_007      15 |    1    1     4    4 |        3        2        2
val_008      20 |    0    1     0    0 |        0        0        0
val_009      11 |    1    3     4    5 |        1        1        0
val_010      14 |    2    0     1    1 |        1        1        1


In [8]:
# --- Cell 9. Save report + verdict ---
report['meta'] = {
    'reranker': 'BAAI/bge-reranker-v2-m3',
    'query_form': f'HyDE passages from Exp-3 (field={HYDE_KEY}), aggregated mean/max over up to 3 passages',
    'candidate_source': 'Exp-3 hyde+enum top-1000 (Qwen3-32B)',
    'n_queries': N_q,
    'baselines': {
        'exp3_hyde+enum_recall@500':   0.376,
        'exp3_hyde+enum_recall@1000':  0.456,
        'exp4_baseline_rrf_recall@50': 0.161,
        'exp4_rerank_de_recall@50':    0.114,
    },
    'win_threshold':    'beat baseline_rrf @50 (0.161)',
    'useful_threshold': 'stat_recall@50 >= 0.30 (sub-25-citation submission)'
}
with open(ART / 'exp_A4b_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

print('=' * 80)
print(f'{"variant":<32} {"@10":>6} {"@20":>6} {"@30":>6} {"@50":>6} {"@100":>6} {"@200":>6} {"@500":>6}')
print('-' * 80)
for k in ['baseline_rrf', 'rerank_de_query', 'rerank_hyde_mean', 'rerank_hyde_max']:
    a = report[k]['agg']
    print(f"{k:<32} {a['stat_recall@10']:>6.3f} {a['stat_recall@20']:>6.3f} "
          f"{a['stat_recall@30']:>6.3f} {a['stat_recall@50']:>6.3f} "
          f"{a['stat_recall@100']:>6.3f} {a['stat_recall@200']:>6.3f} "
          f"{a['stat_recall@500']:>6.3f}")

best = max(report['rerank_hyde_mean']['agg']['stat_recall@50'],
           report['rerank_hyde_max']['agg']['stat_recall@50'])
base = report['baseline_rrf']['agg']['stat_recall@50']
print(f'\nbest HyDE-rerank @50 = {best:.3f}  vs  baseline RRF @50 = {base:.3f}')
print(f'win threshold (beat baseline) -> {"PASS" if best > base else "FAIL — cross-encoder is dead for this task"}')
print(f'useful threshold (>=0.30)     -> {"PASS" if best >= 0.30 else "BELOW"}')

variant                             @10    @20    @30    @50   @100   @200   @500
--------------------------------------------------------------------------------
baseline_rrf                      0.047  0.101  0.121  0.161  0.255  0.315  0.376
rerank_de_query                   0.047  0.087  0.101  0.114  0.148  0.215  0.342
rerank_hyde_mean                  0.087  0.114  0.134  0.174  0.242  0.315  0.369
rerank_hyde_max                   0.074  0.114  0.134  0.174  0.242  0.302  0.369

best HyDE-rerank @50 = 0.174  vs  baseline RRF @50 = 0.161
win threshold (beat baseline) -> PASS
useful threshold (>=0.30)     -> BELOW
